# Stock Sentiment Analysis

This notebook performs sentiment analysis on news articles related to specific stocks and correlates it with stock price movements.

## 1. Setup and Imports

Import necessary libraries and modules from our `src` directory.

In [17]:
import pandas as pd
import sys
import os
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

# Add src directory to path to import modules
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.data_fetcher import get_stock_data, get_news_articles

# Configure pandas display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print("Setup complete.")

Setup complete.


## 2. Define Parameters

Set the stock ticker and date range for analysis.

In [18]:
TICKER = 'AAPL'  # Example: Apple Inc.
END_DATE = datetime.now().strftime('%Y-%m-%d')
# Fetch data for the last 30 days (adjust as needed)
# Note: NewsAPI free tier limits searches to the past month
START_DATE = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d') 

print(f"Ticker: {TICKER}")
print(f"Start Date: {START_DATE}")
print(f"End Date: {END_DATE}")

Ticker: AAPL
Start Date: 2025-03-31
End Date: 2025-04-30


## 3. Fetch Data

Use the functions from `data_fetcher.py` to get stock prices and news articles.

In [19]:
# Fetch Stock Data
print("Fetching stock data...")
stock_df = get_stock_data(TICKER, START_DATE, END_DATE)

if stock_df is not None:
    print(f"Successfully fetched {len(stock_df)} days of stock data.")
    display(stock_df.head())
else:
    print("Failed to fetch stock data.")

Fetching stock data...
Successfully fetched 21 days of stock data.


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2025-03-31,217.009995,225.619995,216.229996,222.130005,65299300,0.0,0.0
1,2025-04-01,219.809998,223.679993,218.899994,223.190002,36412700,0.0,0.0
2,2025-04-02,221.320007,225.190002,221.020004,223.889999,35905900,0.0,0.0
3,2025-04-03,205.539993,207.490005,201.250000,203.190002,103419000,0.0,0.0
4,2025-04-04,193.889999,199.880005,187.339996,188.380005,125910900,0.0,0.0


In [ ]:
# Fetch News Articles
print("Fetching news articles...")
articles_list = get_news_articles(TICKER, START_DATE, END_DATE)

# Convert the list of articles to a DataFrame
if articles_list is not None:
    news_df = pd.DataFrame(articles_list)
    # Convert publishedAt to datetime and extract date
    if 'publishedAt' in news_df.columns:
        news_df['publishedAt'] = pd.to_datetime(news_df['publishedAt'])
        news_df['date'] = news_df['publishedAt'].dt.date
    else:
        news_df['date'] = None # Handle case where publishedAt might be missing
else:
    news_df = pd.DataFrame() # Create an empty DataFrame if fetching failed

# Now check the DataFrame
if not news_df.empty:
    print(f"Successfully fetched and converted {len(news_df)} news articles to DataFrame.")
    display(news_df[['date', 'title', 'description', 'source']].head()) # Display relevant columns
else:
    print("No news articles found or failed to create DataFrame.")

Fetching news articles...
Found 853 articles for 'AAPL'


AttributeError: 'list' object has no attribute 'empty'

## 4. Sentiment Analysis

Apply sentiment analysis to the fetched news articles.

In [ ]:
from src.sentiment_analyzer import analyze_sentiment
# Check if news_df exists and is not empty
if 'news_df' in locals() and not news_df.empty:
    print(f"Performing sentiment analysis on {len(news_df)} articles...")
    # Combine title and description for better context (handle None values)
    news_df['text_to_analyze'] = news_df['title'].fillna('') + ". " + news_df['description'].fillna('')
    # Apply the sentiment analysis function
    # This might take a while depending on the number of articles and your hardware
    sentiment_results = news_df['text_to_analyze'].apply(lambda x: analyze_sentiment(x) if pd.notna(x) else (None, None, None))
    # Unpack results into separate columns
    news_df['sentiment_label'] = sentiment_results.apply(lambda x: x[0])
    news_df['sentiment_score'] = sentiment_results.apply(lambda x: x[1])
    news_df['sentiment_scores_all'] = sentiment_results.apply(lambda x: x[2])
    # Display the results
    print("Sentiment analysis complete.")
    display(news_df[['date', 'title', 'sentiment_label', 'sentiment_score']].head())
    # Display value counts for sentiment labels
    print("\nSentiment Label Distribution:")
    print(news_df['sentiment_label'].value_counts())
else:
    print("Skipping sentiment analysis as no news articles were successfully fetched or the DataFrame is empty.")

Skipping sentiment analysis as no news articles were successfully fetched or the DataFrame is empty.
